In [ ]:
import os
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, ResNet152, ResNet101
from tensorflow.keras.applications.resnet import preprocess_input
from skimage import exposure
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray
import pandas as pd
from skimage.filters import gabor_kernel
from scipy import ndimage as ndi
from scipy.fftpack import dct
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

In [ ]:
import os
import tensorflow as tf
import pandas as pd
from PIL import Image
from tensorflow.keras.applications import ResNet152
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras.layers import GlobalMaxPooling2D, Input
from tensorflow.keras.models import Model

# -------------------------------------------
# ✅ EuPea Attention Module
# -------------------------------------------
class EuPea(tf.keras.layers.Layer):
    def __init__(self, epsilon=1e-5):
        super(EuPea, self).__init__()
        self.epsilon = epsilon

    def call(self, x):
        x_mean = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        x_std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True)

        d = tf.sqrt(tf.square(x - x_mean) + self.epsilon)
        d_min = tf.reduce_min(d, axis=[1, 2], keepdims=True)
        d_max = tf.reduce_max(d, axis=[1, 2], keepdims=True)
        d_norm = (d - d_min) / (d_max - d_min + self.epsilon)

        z = tf.square((x - x_mean) / (x_std + self.epsilon))
        z_min = tf.reduce_min(z, axis=[1, 2], keepdims=True)
        z_max = tf.reduce_max(z, axis=[1, 2], keepdims=True)
        z_norm = (z - z_min) / (z_max - z_min + self.epsilon)

        attention = tf.nn.sigmoid(d_norm + z_norm)
        return x * attention

# -------------------------------------------
# ✅ Build ResNet-152 + EuPea + GMP
# -------------------------------------------
def build_resnet152_eupea():
    base_model = ResNet152(weights='imagenet', include_top=False)

    inp = Input(shape=(224, 224, 3))
    x = base_model(inp, training=False)

    # APPLY EuPea HERE
    x = EuPea()(x)

    # APPLY Global Max Pooling
    x = GlobalMaxPooling2D()(x)

    model = Model(inputs=inp, outputs=x)
    return model

resnet_152 = build_resnet152_eupea()

# -------------------------------------------
# Directory setup
# -------------------------------------------
base_dir = r"/kaggle/input/datasets/saurabhsainiji/iarc-trained-balanced-test-untouched/IARC-casewise-spilit-train-balanced-test-untouched/Balancing-226/Fold-wise/Fold-3"

train_dir = os.path.join(base_dir, "Training")
test_dir = os.path.join(base_dir, "Testing")

labels = ['Type - 1', 'Type - 2', 'Type - 3']

# -------------------------------------------
# ✅ Function to Check Corrupted Images
# -------------------------------------------
def is_valid_image(img_path):
    try:
        img = Image.open(img_path)
        img.verify()
        return True
    except Exception:
        return False

# -------------------------------------------
# Feature extraction function
# -------------------------------------------
def extract_features_from_directory(data_dir):

    features_list = []
    labels_list = []
    filenames_list = []

    corrupted_count = 0
    processed_count = 0

    for label_idx, label in enumerate(labels):

        label_path = os.path.join(data_dir, label)

        if not os.path.exists(label_path):
            print(f"⚠️ Warning: directory not found {label_path}")
            continue

        print(f"\n📂 Processing Class: {label}")

        for root, _, files in os.walk(label_path):

            for file in files:

                # Only image files
                if not (
                    file.lower().endswith('.jpg')
                    or file.lower().endswith('.jpeg')
                    or file.lower().endswith('.png')
                ):
                    continue

                img_path = os.path.join(root, file)

                # ----------------------------------
                # Skip corrupted images
                # ----------------------------------
                if not is_valid_image(img_path):
                    corrupted_count += 1
                    continue

                try:

                    img = tf.keras.preprocessing.image.load_img(
                        img_path,
                        target_size=(224, 224)
                    )

                    x = tf.keras.preprocessing.image.img_to_array(img)

                    x = tf.expand_dims(x, axis=0)

                    x = preprocess_input(x)

                    # --- Extract features ---
                    features = resnet_152.predict(
                        x,
                        verbose=0
                    )[0]

                    features_list.append(features)
                    labels_list.append(label_idx)
                    filenames_list.append(file)

                    processed_count += 1

                    if processed_count % 50 == 0:
                        print(f"✅ Processed Images: {processed_count}")

                except Exception:
                    corrupted_count += 1
                    continue

    print("\n======================================")
    print(f"✅ Total Processed Images : {processed_count}")
    print(f"❌ Total Corrupted Images: {corrupted_count}")
    print("======================================")

    df = pd.DataFrame({
        'image_name': filenames_list,
        'features': features_list,
        'labels': labels_list
    })

    return df


# -------------------------------------------
# Extract training & testing features
# -------------------------------------------
res_152_train_features_df = extract_features_from_directory(train_dir)
res_152_test_features_df = extract_features_from_directory(test_dir)

print("Training features shape:", res_152_train_features_df.shape)
print("Testing features shape:", res_152_test_features_df.shape)
print(res_152_train_features_df.head())

# -------------------------------------------
# Save results
# -------------------------------------------
res_152_train_features_df.to_pickle("resnet152_train_features.pkl")
res_152_test_features_df.to_pickle("resnet152_test_features.pkl")

print("\n✅ Feature files saved successfully!")

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, f1_score, accuracy_score

# --- ✅ Separate features, labels, and image names for training and testing sets ---
X_train = np.stack(res_152_train_features_df['features'].values)
y_train = np.array(res_152_train_features_df['labels'].values)

X_test = np.stack(res_152_test_features_df['features'].values)
y_test = np.array(res_152_test_features_df['labels'].values)
image_names = np.array(res_152_test_features_df['image_name'].values)  # store filenames

print("ResNet50 features with SVM (Train/Test evaluation)")

# --- ✅ Initialize and Train SVM ---
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

# --- ✅ Predict on test data ---
y_pred = svm.predict(X_test)

# --- ✅ Compute confusion matrix ---
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix)

# --- ✅ Compute metrics ---
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
sensitivity = np.mean([
    conf_matrix[i, i] / conf_matrix[i, :].sum() if conf_matrix[i, :].sum() > 0 else 0
    for i in range(conf_matrix.shape[0])
])
specificity = np.mean([
    (np.sum(conf_matrix) - np.sum(conf_matrix[i, :]) - np.sum(conf_matrix[:, i]) + conf_matrix[i, i]) /
    (np.sum(conf_matrix) - np.sum(conf_matrix[:, i])) if np.sum(conf_matrix[:, i]) > 0 else 0
    for i in range(conf_matrix.shape[0])
])
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

# --- ✅ Print overall results ---
print(f"\nAccuracy: {accuracy:.6f}")
print(f"Specificity: {specificity:.6f}")
print(f"Sensitivity (Recall): {sensitivity:.6f}")
print(f"Precision: {precision:.6f}")
print(f"F1 Score: {f1:.6f}")

# --- ✅ Identify and print misclassified images ---
print("\nMisclassified Images:")
misclassified_indices = np.where(y_test != y_pred)[0]

if len(misclassified_indices) == 0:
    print("🎉 No misclassifications! All predictions are correct.")
else:
    for idx in misclassified_indices:
        true_label = labels[y_test[idx]]
        pred_label = labels[y_pred[idx]]
        img_name = image_names[idx]
        print(f"Image: {img_name} | True: {true_label} | Predicted: {pred_label}")
